In [ ]:
%load_ext autoreload
%autoreload 2
import gym
import dotenv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from stable_baselines3 import PPO
from collections import Counter
dotenv.load_dotenv()

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

In [15]:


easy_models_dict = {
    # Behavior 1: 
    1: {'path': "models/fruitbot/20251223-133810_easy/ppo_final.zip", 'index': 1, 'name': 'avoid_walls_random_food'},
    
    # Behavior 2: don't open doors and collect all food
    2: {'path': 'models/fruitbot/20260116-074523_easy/ppo_final.zip', 'index': 2, 'name': 'no_doors_collect_all'},
    
    # Behavior 3: don't open doors and collect only fruits
    3: {'path': "models/fruitbot/20260117-134142_easy/ppo_final.zip", 'index': 3, 'name': 'no_doors_fruits_only'},
    
    # Behavior 4: collect only fruits and open doors
    4: {'path': "models/fruitbot/20251231-174002_easy/ppo_final.zip", 'index': 4, 'name': 'open_doors_fruits_only'},
    
    # Behavior 5: open doors and collect all foods
    5: {'path': "models/fruitbot/20260121-152950_easy/ppo_final.zip", 'index': 5, 'name': 'open_doors_collect_all'},
    
    # Behavior 6: open doors and avoid all foods  
    6: {'path': "models/fruitbot/20260103-073446_easy/ppo_final.zip", 'index': 6, 'name': 'open_doors_avoid_food'},
    
    # Behavior 7: try to open doors and collect only fruits
    7: {'path': "models/fruitbot/20260105-075949_easy/ppo_final.zip", 'index': 7, 'name': 'only_fruits_tries_open_doors'},

    # Behavior 8: do not open doors and collect only junk
    8: {"path": "models/fruitbot/20260116-210051_easy/ppo_final.zip", 'index': 8, 'name': 'no_doors_junk_only'},
}


In [23]:
def evaluate_agent_on_env(env, agent, num_episodes=10) -> dict:
    """Evaluate an agent on a given environment over multiple episodes.
    
    Args:
        env: Gym environment
        agent: Stable Baselines3 agent
        num_episodes: Number of episodes to evaluate
    
    Returns:
        Dict with mean, std, min, max scores and all scores list
    """
    scores = []
    ep_lengths = []
    for episode in range(num_episodes):
        obs= env.reset()
        steps = 0
        done = False
        total_reward = 0
        
        while not done:
            action, _ = agent.predict(obs, deterministic=True)
            action = action.item() if hasattr(action, 'item') else int(action)
            obs, reward, done, info = env.step(action)
            total_reward += reward
            steps += 1
        
        scores.append(total_reward)
        ep_lengths.append(steps)
        wall_hits = len([h for h in ep_lengths if h < max(ep_lengths)])

    return {
        'mean_score': np.mean(scores),
        'std_score': np.std(scores),
        'mean_length': np.mean(ep_lengths),
        'wall_hits': wall_hits/num_episodes,
    }

In [24]:
# Configuration setup
CONFIG_INDEX = 6  # walls_doors d30_g6_b6
NUM_EPISODES = 50  # Each model runs on 50 episodes
FIXED_SEED = 42  # Fixed seed for consistency across models

env_config = dpu_clf.get_env_config_by_index(CONFIG_INDEX, FIXED_SEED)

print(f"Using configuration index {CONFIG_INDEX}:")
print(f"  Config: {env_config}")
env = gym.make("procgen:procgen-fruitbot-v0", **env_config)
path = "..\\models\\fruitbot\\20251223-133810_easy\\ppo_final.zip"
# agent = PPO.load(easy_models_dict[1]['path'][:-4])
agent = PPO.load(path)
base_agent_results = evaluate_agent_on_env(env, agent, 1)
print(f"Base agent results: {base_agent_results}")

Using configuration index 6:
  Config: {'distribution_mode': 'easy', 'food_diversity': 6, 'use_discrete_action_wrapper': True, 'use_stay_bonus_wrapper': False, 'fruitbot_reward_positive': 1.0, 'fruitbot_reward_negative': -1.0, 'fruitbot_reward_wall_hit': -3.0, 'fruitbot_reward_completion': 10.0, 'fruitbot_reward_step': 0.0, 'fruitbot_num_walls': 3, 'fruitbot_force_no_walls': False, 'fruitbot_num_good_range': 1, 'fruitbot_num_bad_range': 1, 'fruitbot_wall_gap_pct': 40, 'fruitbot_num_good_min': 6, 'fruitbot_num_bad_min': 6, 'fruitbot_door_prob_pct': 40, 'rand_seed': 42, 'start_level': 42}
Using prebuilt binaries from: c:\users\matan\master_thesis\rl_envs\procgen\procgen\.build\relwithdebinfo\RelWithDebInfo
Base agent results: {'mean_score': 19.0, 'std_score': 0.0, 'mean_length': 132.0, 'wall_hits': 0.0}


## Collect the Data

In [ ]:
better_app_path = "..\\models\\fruitbot\\20251231-174002_easy\\ppo_final.zip"
